# Tranformation strategy

## Bronze append

In [0]:
%sql
USE CATALOG adb_std;
CREATE SCHEMA IF NOT EXISTS adb_std.bronze;

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;
-- DESCRIBE EXTERNAL LOCATION ext-
-- path = "abfss://iot@adls2tdstudent.dfs.core.windows.net/"

In [0]:
%sql
-- DROP SCHEMA IF EXISTS adb_std.bronze CASCADE;
-- CREATE SCHEMA IF NOT EXISTS adb_std.bronze;

-- CREATE THE TABLE WITH EXTERNAL LOCATION FIRST
CREATE TABLE IF NOT EXISTS adb_std.bronze.iot_raw(
    _ingested_at TIMESTAMP,
    _source_file STRING,
    device_id STRING,
    device_name STRING,
    model STRING,
    currentLocation STRING,
    PM10 STRING,
    PM2_5 STRING,
    Temperature STRING,
    rel_Luftfeuchte STRING,
    Luftdruck STRING,
    _rescued_data STRING
)
USING DELTA
LOCATION 'abfss://iot@adls2tdstudent.dfs.core.windows.net/bronze/iot_raw';

In [0]:
%sql
DROP TABLE IF EXISTS adb_std.bronze.iot_raw ;

In [0]:
from pyspark.sql import functions as F

In [0]:
from pyspark.sql import functions as F
# 1. storage & checkpoint paths inside external locations
landing_path = "/Volumes/adb_std/default/raw_iot/opensense"
bronze_path = "/Volumes/adb_std/bronze/iot_raw" # raw path of external table bronze
checkpoint_path = "/Volumes/adb_std/default/raw_iot/processed/opensense/checkpoint"

# 2. Ingest raw IOT
# because delta table have conventional name, need column name change
# mapping_column = {
#     `PM2.5`: "PM2_5",
#     "PM10": "PM10",
#     "Temperatur": "Temperature",
#     "rel. Luftfeuchte" : "rel_Luftfeuchte"
# }
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    # .option("cloudFiles.useNotifications", "true")
    .option("cloudFiles.maxFilesPerTrigger", 500)
    # 2. Pack multiple small files into larger in-memory partitions (e.g., 64MB - 128MB)
    .option("cloudFiles.maxBytesPerTrigger", "10g")
    .option("cloudFiles.schemaLocation", "/Volumes/adb_std/default/raw_iot/processed/opensense/iot_raw/schema")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .load(landing_path)
)
raw_stream_renamed = raw_stream\
    .withColumnRenamed("PM2.5", "PM2_5")\
    .withColumnRenamed("Temperatur", "Temperature")\
    .withColumnRenamed("rel. Luftfeuchte", "rel_Luftfeuchte")\
    .withColumn("_ingested_at", F.current_timestamp())\
    .withColumn("_source_file", F.col("_metadata.file_path") )

# 3. append to delta table
# toTable is for table empty created in sql
bronze_write_query = (
    raw_stream_renamed
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow = True)
    .toTable("adb_std.bronze.iot_raw")
    # .start(bronze_path)
)

bronze_write_query.awaitTermination()



In [0]:
%sql
SELECT *
FROM adb_std.bronze.iot_raw
ORDER BY `_ingested_at` DESC
LIMIT 10;

In [0]:
%sql
ALTER TABLE adb_std.bronze.iot_raw CLUSTER BY (device_id, _ingested_at);

In [0]:
print(bronze_write_query.recentProgress)

In [0]:
%sql
DESCRIBE EXTENDED adb_std.bronze.iot_raw;